Phase 3: Neural Network Interpretability and Motif Extraction. Extract embeddings from sentence transformers, cluster Asian/SE Asian folktales using k-means (or SAE-lens when available), extract narrative motifs and archetypes from each cluster, and analyze distinctive features, n-grams, and representative stories per cluster.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import yaml
import logging
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import FolktaleDataLoader
from src.model import FolktaleClassifier
from src.interpretability import MotifExtractor
from src.visualization import (
    plot_cluster_sizes,
    plot_tsne_clusters
)

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

# Load configuration
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print('Setup complete.')

Load Data and Trained Classifier

In [ ]:
# Load data
loader = FolktaleDataLoader('../config/config.yaml')
asian_df = loader.load_asian_tales()

if len(asian_df) > 0:
    # Preprocess texts
    asian_df['text_processed'] = asian_df['text'].apply(loader.preprocess_text)
    texts_asian = asian_df['text_processed'].values
    
    print(f"Loaded {len(texts_asian)} Asian/SE Asian folktales")
    
    # Load trained classifier
    clf = FolktaleClassifier(config, device='cpu')
    try:
        clf.load('../results/models/folktale_classifier.pkl')
        print("✓ Classifier loaded successfully")
    except FileNotFoundError:
        print("Error: Trained classifier not found. Run notebook 02 first.")
else:
    print("No Asian folktales available. Cannot proceed with clustering.")

Extract Embeddings from Sentence Transformer

In [ ]:
if len(asian_df) > 0:
    print("Extracting embeddings from sentence transformer...")
    embeddings = clf.encode_texts(texts_asian)
    
    print(f"\nEmbedding shape: {embeddings.shape}")
    print(f"  - Number of texts: {embeddings.shape[0]}")
    print(f"  - Embedding dimension: {embeddings.shape[1]}")
    
    # Save embeddings for later use
    output_dir = Path('../results/embeddings')
    output_dir.mkdir(parents=True, exist_ok=True)
    np.save(str(output_dir / 'asian_folktales.npy'), embeddings)
    print(f"\n✓ Embeddings saved to {output_dir / 'asian_folktales.npy'}")

Cluster Embeddings using K-means

In [ ]:
if len(asian_df) > 0:
    # Initialize motif extractor
    motif_extractor = MotifExtractor(config)
    
    # Cluster embeddings
    n_clusters = config['clustering']['kmeans']['n_clusters']
    print(f"Clustering with n_clusters={n_clusters}...")
    
    cluster_assignments, cluster_info = motif_extractor.cluster_embeddings(
        embeddings,
        method='kmeans',
        n_clusters=n_clusters
    )
    
    print(f"\nClustering Results:")
    print(f"  Inertia: {cluster_info['inertia']:.2f}")
    
    # Visualize cluster sizes
    plot_cluster_sizes(cluster_assignments, '../results/plots/03_cluster_sizes.png')

In [ ]:
# Visualize clusters in 2D via t-SNE
print("\nGenerating t-SNE visualization...")
try:
    plot_tsne_clusters(
        embeddings,
        cluster_assignments,
        output_path='../results/plots/03_tsne_clusters.png'
    )
except Exception as e:
    print(f"Warning: t-SNE visualization failed: {e}")

Extract Narrative Motifs: Top-Activating Tokens

In [ ]:
if len(asian_df) > 0:
    # Extract top tokens for each cluster
    print("Extracting top tokens per cluster...")
    top_tokens = motif_extractor.extract_top_tokens(
        embeddings,
        texts_asian,
        cluster_assignments
    )
    
    # Display results
    for cluster_id in sorted(top_tokens.keys()):
        tokens = top_tokens[cluster_id]
        cluster_size = np.sum(cluster_assignments == cluster_id)
        print(f"Cluster {cluster_id} ({cluster_size} stories):")
        print("  Top tokens (word-frequency motif indicators):")
        for token, count in tokens[:10]:  # Top 10
            print(f"    - {token}: {count}")

Extract Representative Stories per Cluster

In [ ]:
if len(asian_df) > 0:
    # Extract representative stories
    print("Extracting representative stories per cluster...")
    rep_stories = motif_extractor.extract_representative_stories(
        embeddings,
        texts_asian,
        asian_df,
        cluster_assignments
    )
    
    for cluster_id in sorted(rep_stories.keys()):
        stories = rep_stories[cluster_id]
        print(f"Cluster {cluster_id}:")
        for i, story in enumerate(stories, 1):
            title = story.get('title', 'Unknown')
            region = story.get('region', 'Unknown')
            distance = story.get('distance_to_centroid', 0)
            print(f"  {i}. {title} [{region}] (distance: {distance:.3f})")
            # Print excerpt
            excerpt = story['text'][:200].replace('\n', ' ')
            print(f"     Excerpt: {excerpt}...\n")

In [ ]:
# Extract distinctive n-grams
print("Extracting distinctive n-grams per cluster...")
ngrams = motif_extractor.extract_ngrams(
    texts_asian,
    cluster_assignments
)

for cluster_id in sorted(ngrams.keys()):
    ngram_list = ngrams[cluster_id]
    print(f"Cluster {cluster_id} - Top narrative phrases:")
    for ngram, score in ngram_list[:8]:  # Top 8
        print(f"  - \'{ngram}\': {score:.4f}")

Motif Analysis and Archetype Naming

In [ ]:
print("Based on cluster analysis, proposed archetypes for Asian/SE Asian folktales:")
print("(Review the top tokens, n-grams, and representative stories above to")
print("verify these inferred narrative patterns)")

In [ ]:
# Save motif analysis results
results = {
    'cluster_assignments': cluster_assignments.tolist(),
    'cluster_info': cluster_info,
    'top_tokens': {str(k): [(t, int(c)) for t, c in v] for k, v in top_tokens.items()},
    'archetype_proposals': {
        str(k): {
            'name': v['name'],
            'key_tokens': v['key_tokens'],
            'cluster_size': v['cluster_size']
        }
        for k, v in archetype_proposals.items()
    }
}

import json
output_dir = Path('../results/analysis')
output_dir.mkdir(parents=True, exist_ok=True)

with open(str(output_dir / '03_motif_analysis.json'), 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✓ Motif analysis saved to {output_dir / '03_motif_analysis.json'}")

Summary and Recommendations

In [ ]:
print(f"Extracted embeddings from {len(embeddings)} Asian/SE Asian folktales")
print(f"Clustered into {n_clusters} narrative archetypes")
print(f"Extracted motif indicators (top tokens, n-grams)")
print(f"Identified representative stories per cluster")
print(f"Proposed archetype names based on distinctive features")
print("NEXT STEPS")
print("1. Manual Verification: Review the inferred archetypes against representative stories to refine cluster interpretations.")
print("2. Motif Refinement: Cross-reference tokens/n-grams with literature on Buddhist, Confucian, and religious narrative patterns.")
print("3. ATU Comparison: Compare discovered archetypes to existing ATU categories and identify gaps/improvements.")
print("4. Build Taxonomy: Design a parallel classification system for East/Southeast Asian folktales based on these archetypes.")
print("All results saved in: ../results/")